In [0]:
%pip install databricks-feature-engineering
dbutils.library.restartPython()

In [0]:
import logging
from databricks.feature_engineering import FeatureEngineeringClient,FeatureLookup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import os
import pickle
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from mlflow.models import infer_signature

In [0]:
logger = logging.getLogger("ChallengerModelSelection")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter(
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)
logger.propagate = False

In [0]:
class ModelSelection:

    def __init__(self,spark,experiment_name,model_name,model_artifact_name):

        self.spark = spark
        self.experiment_name = experiment_name
        self.model_name = model_name
        self.model_artifact_name = model_artifact_name

        logger.info("Model Selection initialized")
   
    def get_best_run(self):

        logger.info("Searching for best model based on validation R2")

        experiment = mlflow.get_experiment_by_name(self.experiment_name)

        if experiment is None:
            raise ValueError(f"Experiment not found: {self.experiment_name}")

        runs = mlflow.search_runs(
            experiment_ids=[experiment.experiment_id],
            filter_string="status = 'FINISHED' and run_name='DecisionTree_Training'",
            order_by=["metrics.validation_r2 DESC"],
            max_results=1
        )
        if runs.empty:
            raise ValueError("No completed runs with validation R2 found")

        best_run = runs.iloc[0]

        logger.info("Best run ID: %s",best_run["run_id"])

        logger.info("Best run name: %s",best_run["tags.mlflow.runName"])

        logger.info("Best validation R2: %.4f",best_run["metrics.validation_r2"])
        return best_run
        
    def register_model(self,best_run):

        logger.info("Registering best model in Unity Catalog")
        run_id = best_run["run_id"]
        model_uri = f"runs:/{run_id}/{self.model_artifact_name}"
        logger.info("Model URI: %s",model_uri)
        mlflow.set_registry_uri("databricks-uc")
        reg_model = mlflow.register_model(
            model_uri=model_uri,
            name=self.model_name
        )
        logger.info(
            "Model registered successfully. Version: %s",
            reg_model.version
        )
        return reg_model
    
    def run(self):
        logger.info("Challenger model selection started")
        best_run = self.get_best_run()
        reg_model = self.register_model(best_run)
        logger.info("Challenger model selection completed")
        return True

In [0]:
feature_client = FeatureEngineeringClient()
bst_model = ModelSelection(
    spark=spark,
    experiment_name = "/Users/rovindcosta133@gmail.com/Demand_Prediction",
    model_name="oag.ml.demand_prediction_model",
    model_artifact_name="decision_tree_model"
)
bst_model.run()


